PIFOCast, a simple weather prediction model
===========================================


##### Copyright 2025 Nicolas Gasnier.

In [ ]:
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

##### Introduction

This project is based on the following :
- Google Deepmind [GraphCast](https://www.science.org/doi/10.1126/science.adi2336) and [GenCast](https://arxiv.org/abs/2312.15796) papers, and associated [source code](https://github.com/google-deepmind/graphcast) as a source of inspiration
- Tensorflow GNN's colab example ["Learning shortest path with GraphNetowks in TF-GNN"](https://colab.research.google.com/github/tensorflow/gnn/blob/master/examples/notebooks/graph_network_shortest_path.ipynb#scrollTo=qr1_8ttC08vu), from wich the base Encoder / Processor / Decoder architecture is derived.


In [ ]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import math
import collections
import functools
import xarray as xr
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_gnn as tfgnn
from tensorflow_gnn import runner
from typing import Callable, Optional, Mapping

#### The model

### MLP utility for feature encoding, processing and decoding

Note that gradients vanishes with relu, leaky_relu is more robust but the silu function seems to work best - it is used in Gencast after all. So we default to that.

In [ ]:
def build_mlp(num_hidden_layers: int,
              hidden_size: int,
              output_size: int,
              use_layer_norm: bool,
              activation: str = "silu",#tf.keras.layers.LeakyReLU(alpha=0.01), 
              activate_final: bool = False,
              name: str = "mlp"):
    """Builds an MLP."""
    output_sizes = [hidden_size] * num_hidden_layers + [output_size]
    mlp = tf.keras.Sequential(name="mlp")
    for layer_i, size in enumerate(output_sizes):
        layer_activation = activation
        if not activate_final and layer_i == len(output_sizes) - 1:
            layer_activation = None
        mlp.add(tf.keras.layers.Dense(
            size,
            activation=layer_activation,
            use_bias=True,
            kernel_initializer="variance_scaling",
            bias_initializer="zeros",
            name=f"{name}/dense_{layer_i}"))
    if use_layer_norm:
        mlp.add(tf.keras.layers.LayerNormalization(
            name=f"{name}/layer_norm"))
    return mlp

### Graph structure definition
We define the graph as a lat/lon grid :
- nodes are the grid points, with features a vector of the meteorological + forcings fields for that grid point,
- edges connects the grid points, with no features for the moment. We made edges in both directions, and made it wrap around at Greenwich. For poles, we don't do any wrapping yet. 

In [ ]:
class LatLonGrid:
    def __init__(self, from_grib, regrid_factor=4):
        """
        Parameters :
        from_grib : grib file to take as example of lat lon data
        regrid_factor : grib slicing factor to reduce resolution
        """
        ds = xr.load_dataset(from_grib, engine="cfgrib", decode_timedelta=False)
        ds_reduced = ds.isel(latitude=slice(0, None, regrid_factor), longitude=slice(0, None, regrid_factor))

        self.NLat = ds_reduced.latitude.shape[0]
        self.NLon = ds_reduced.longitude.shape[0]
        self.NNodes = self.NLat * self.NLon

        self.lat2d = tf.reshape(tf.convert_to_tensor(ds_reduced.latitude.expand_dims({"longitude": self.NLon}, axis=1), dtype_hint="float"), [self.NNodes])
        self.lon2d = tf.reshape(tf.convert_to_tensor(ds_reduced.longitude.expand_dims({"latitude": self.NLat}, axis=0), dtype_hint="float"), [self.NNodes])

        self.__buildAdjacency(ds_reduced)

    def __calcEdgeLength(self, ds):
        # Calc length of edges between nodes on meridian between each latitude
        med_length = []
        for j in range(self.NLat-1): 
            med_length.append(
                self.distanceLatLon(float(ds.latitude[j]), 0, 
                                    float(ds.latitude[j+1]),0))
            
        # Calc length of edges between nodes on each latitude circule
        lat_length = []
        for j in range(self.NLat): 
            lat_length.append(
                self.distanceLatLon(float(ds.latitude[j]), float(ds.longitude[0]), 
                                    float(ds.latitude[j]), float(ds.longitude[1])
                )
            )

        return med_length, lat_length


    def __buildAdjacency(self, ds):

        med_length, lat_length = self.__calcEdgeLength(ds)

        sources = []
        targets = []
        lengthes = []
        coords_x = []
        coords_y = []
        # Nodes for poles are note linked around latitude cirles
        node_id = self.NLon # Skip the first row of pole values
        for j in range(1, self.NLat-1): 
            for i in range(self.NLon):                
                # Get data from the right
                if i<self.NLon-1:
                    sources.append(node_id+1)
                    targets.append(node_id)
                    lengthes.append(lat_length[j])
                else:
                    sources.append(node_id-self.NLon+1)
                    targets.append(node_id)
                    lengthes.append(lat_length[j])
                coords_x.append(1)
                coords_y.append(0)

                # Get data from the top (higher lats indexed first)
                sources.append(node_id-self.NLon)
                targets.append(node_id)
                lengthes.append(med_length[j-1])
                coords_x.append(0)
                coords_y.append(1)

                # Get data from the left
                if i==0: # Wrap at greenwich
                    sources.append(node_id+self.NLon-1)
                    targets.append(node_id)
                    lengthes.append(lat_length[j])
                else:
                    sources.append(node_id-1)
                    targets.append(node_id)
                    lengthes.append(lat_length[j])
                coords_x.append(-1)
                coords_y.append(0)

                # Get data from the bottom
                sources.append(node_id+self.NLon)
                targets.append(node_id)
                lengthes.append(med_length[j])
                coords_x.append(0)
                coords_y.append(-1)

                node_id += 1
            
            # prout = len (lengthes)-1
            # print("lat=", float(ds.latitude[j]), " : ")
            # print("  right->[", sources[prout-3], ", ",targets[prout-3],"]", lengthes[prout-3])
            # print("  top<-[", sources[prout-2], ", ",targets[prout-2],"]", lengthes[prout-2])
            # print("  left^^[", sources[prout-1], ", ",targets[prout-1],"]", lengthes[prout-1])
            # print("  bottomvv[", sources[prout], ", ",targets[prout],"]", lengthes[prout])
    

        self.sources = sources
        self.targets = targets
        self.lengthes = lengthes
        self.coords_x = coords_x
        self.coords_y = coords_y
        self.NEdges = len(sources)

    
    
    def distanceLatLon(self, lata, lona, latb, lonb):
        lat1 = lata/180*math.pi
        lon1 = lona/180*math.pi
        lat2 = latb/180*math.pi
        lon2 = lonb/180*math.pi

        d = math.acos(math.sin(lat1)*math.sin(lat2) + math.cos(lat1)*math.cos(lat2)*math.cos(lon2-lon1))
        return d*6371598 # sphere de Picard
                        # 6378137 sphere IAG-GRS80


In [ ]:

def buildGridGNN(X, X_edges, grid:LatLonGrid, **task_kwargs):
    """
    Parameters : 
    X : [NLat x NLon, NFeatures], the input data 
    Y : [NLat x NLon, NFeatures], the expected output data (for learning)
    latitudes : array of NLat latitudes
    longitudes : array of NLon longitudes
    """
   
    graph = tfgnn.GraphTensor.from_pieces(
        node_sets={
            "grid": tfgnn.NodeSet.from_fields(
                sizes=tf.constant([ grid.NNodes ]),
                features={
                    "features": X
                }
            )
        },
        edge_sets={
            "edges": tfgnn.EdgeSet.from_fields(
                sizes=tf.constant([ grid.NEdges ]),
                features={
                    "edge_features": X_edges
                },
                adjacency=tfgnn.Adjacency.from_indices(
                    source=("grid", tf.constant(grid.sources)),
                    target=("grid", tf.constant(grid.targets))
                )
            )
        }
    )

    return graph

### Encoder - Processor - Decoder architecture


In [ ]:
def GraphNetworkGraphUpdate(
        *,
        edges_next_state_factory: Callable[..., tf.keras.layers.Layer],
        nodes_next_state_factory: Callable[..., tf.keras.layers.Layer],
        context_next_state_factory: Optional[Callable[..., tf.keras.layers.Layer]],
        receiver_tag: Optional[tfgnn.IncidentNodeTag] = tfgnn.TARGET,
        reduce_type_to_nodes: str = "sum",
        reduce_type_to_context: str = "sum",
        use_input_context_state: bool = True,
        name: str = "graph_network"):
    """Returns a GraphUpdate to run a GraphNetwork on all node sets and edge sets.

    The returned layer implements a Graph Network, as described by
    Battaglia et al.: ["Relational inductive biases, deep learning, and
    graph networks"](https://arxiv.org/abs/1806.01261), 2018, generalized
    to heterogeneous graphs.

    It expects an input GraphTensor with a `tfgnn.HIDDEN_STATE` feature on all
    node sets and edge sets, and also context if `use_input_context_state=True`.
    It runs edge, node, and context updates, in this order, separately for each
    edge set, node set (regardless whether it has an incoming edge set), and also
    context if `context_next_state_factory` is set. Finally, it returns a
    GraphTensor with updated hidden states, incl. a context state, if
    `context_next_state_factory` is set.

    The model can also behave as an Interaction Network ([Battaglia et al., NIPS
    2016](https://proceedings.neurips.cc/paper/2016/hash/3147da8ab4a0437c15ef51a5cc7f2dc4-Abstract.html))
    by setting
        * `use_input_context_state = False`
        * `context_next_state_factory = None`

    Args:
        edges_next_state_factory: Called with keyword argument `edge_set_name=`
        for each edge set to return the NextState layer for use in the respective
        `tfgnn.keras.layers.EdgeSetUpdate`.
        nodes_next_state_factory: Called with keyword argument `node_set_name=`
        for each node set to return the NextState layer for use in the respective
        `tfgnn.keras.layers.NodeSetUpdate`.
        context_next_state_factory: If set, a `tfgnn.keras.layers.ContextUpdate`
        is included with the NextState layer returned by calling this factory.
        receiver_tag: The incident node tag at which each edge set is used to
        update node sets. Defaults to `tfgnn.TARGET`.
        reduce_type_to_nodes: Controls how incident edges at a node are aggregated
        within each EdgeSet. Defaults to `"sum"`. (The aggregates of the various
        incident EdgeSets are then concatenated.)
        reduce_type_to_context: Controls how the nodes of a NodeSet or the edges of
        an EdgeSet are aggregated for the context update. Defaults to `"sum"`.
        (The aggregates of the various NodeSets/EdgeSets are then concatenated.)
        use_input_context_state: If true, the input `GraphTensor.context` must have
        a `tfgnn.HIDDEN_STATE` feature that gets used as input in all edge, node
        and context updates.
        name: A name for the returned layer.
    """
    def deferred_init_callback(graph_spec):
        context_input_feature = (
            tfgnn.HIDDEN_STATE if use_input_context_state else None)

        # To keep track node types that receive each edge type.
        incoming_edge_sets = collections.defaultdict(list)

        # Used to filter out auxiliary readouts nodes and edeges
        def isNotAuxiliary(n):
            return not n.startswith("_")

        # For every edge set, create an EdgeSetUpdate.
        edge_set_updates = {}
        for edge_set_name in sorted(graph_spec.edge_sets_spec.keys()): #filter(isNotAuxiliary, sorted(graph_spec.edge_sets_spec.keys()))
            next_state = edges_next_state_factory(edge_set_name=edge_set_name)
            edge_set_updates[edge_set_name] = tfgnn.keras.layers.EdgeSetUpdate(
                next_state=next_state,
                edge_input_feature=tfgnn.HIDDEN_STATE,
                node_input_feature=tfgnn.HIDDEN_STATE,
                context_input_feature=context_input_feature)
        # Keep track of which node set is the receiver for this edge type
        # as we will need it later.
        target_name = graph_spec.edge_sets_spec[
            edge_set_name].adjacency_spec.node_set_name(receiver_tag)
        incoming_edge_sets[target_name].append(edge_set_name)

        # For every node set, create a NodeSetUpdate.
        node_set_updates = {}
        for node_set_name in sorted(graph_spec.node_sets_spec.keys()): #filter(isNotAuxiliary, sorted(graph_spec.node_sets_spec.keys()))
            # Apply a node update, after summing *all* of the received edges
            # for that node set.
            next_state = nodes_next_state_factory(node_set_name=node_set_name)
            node_set_updates[node_set_name] = tfgnn.keras.layers.NodeSetUpdate(
                next_state=next_state,
                edge_set_inputs={
                    edge_set_name: tfgnn.keras.layers.Pool(
                        receiver_tag, reduce_type_to_nodes,
                        feature_name=tfgnn.HIDDEN_STATE)
                    for edge_set_name in incoming_edge_sets[node_set_name]},
                node_input_feature=tfgnn.HIDDEN_STATE,
                context_input_feature=context_input_feature)

        # Create a ContextUpdate, if requested.
        context_update = None
        if context_next_state_factory is not None:
            next_state = context_next_state_factory()
            context_update = tfgnn.keras.layers.ContextUpdate(
                next_state=next_state,
                edge_set_inputs={
                    edge_set_name: tfgnn.keras.layers.Pool(
                        tfgnn.CONTEXT, reduce_type_to_context,
                        feature_name=tfgnn.HIDDEN_STATE)
                    for edge_set_name in sorted(graph_spec.edge_sets_spec.keys())}, #ilter(isNotAuxiliary(sorted(graph_spec.edge_sets_spec.keys()))
                node_set_inputs={
                    node_set_name: tfgnn.keras.layers.Pool(
                        tfgnn.CONTEXT, reduce_type_to_context,
                        feature_name=tfgnn.HIDDEN_STATE)
                    for node_set_name in sorted(graph_spec.node_sets_spec.keys())}, #filter(isNotAuxiliary(sorted(graph_spec.node_sets_spec.keys()))
                context_input_feature=context_input_feature)
        return dict(edge_sets=edge_set_updates,
                    node_sets=node_set_updates,
                    context=context_update)

    return tfgnn.keras.layers.GraphUpdate(
        deferred_init_callback=deferred_init_callback, name=name)


class PifoEncodeProcessDecode(tf.keras.layers.Layer):
    def __init__(
        self,
        edge_output_size: Optional[int],
        node_output_size: Optional[int],
        context_output_size: Optional[int],
        num_message_passing_steps: int,
        num_mlp_hidden_layers: int,
        mlp_hidden_size: int,
        latent_size: int,
        use_layer_norm: bool,
        shared_processors: bool,
        reduce_type_to_nodes: str = "sum",
        reduce_type_to_context: str = "sum",
        name: str = "encode_process_decode"):
        super().__init__(name=name)
        
        # Build graph encoder.
        def encoder_fn(graph_piece, *, edge_set_name=None, node_set_name=None):
            piece_name = (f"edges_{edge_set_name}" if edge_set_name else
                            f"nodes_{node_set_name}" if node_set_name else "context")
            mlp = build_mlp(num_hidden_layers=num_mlp_hidden_layers,
                            hidden_size=mlp_hidden_size,
                            output_size=latent_size,
                            use_layer_norm=use_layer_norm,
                            name=f"{self.name}/encoder/{piece_name}")
            return mlp(graph_piece[tfgnn.HIDDEN_STATE])

        self._encoder = tfgnn.keras.layers.MapFeatures(
            edge_sets_fn=encoder_fn, node_sets_fn=encoder_fn, context_fn=(encoder_fn if context_output_size else None))

        # Build graph processor(s).
        # We will just concatenate all inputs to each edge update, node update
        # and context upcate, and run an MLP on it.

        def processor_fn(*, processor_name, edge_set_name=None, node_set_name=None):
            if edge_set_name is not None:
                mlp_name = f"{processor_name}/edges_{edge_set_name}"
            elif node_set_name is not None:
                mlp_name = f"{processor_name}/nodes_{node_set_name}"
            else:
                mlp_name = f"{processor_name}/context"
            mlp = build_mlp(name=mlp_name,
                            num_hidden_layers=num_mlp_hidden_layers,
                            hidden_size=mlp_hidden_size,
                            output_size=latent_size,
                            use_layer_norm=use_layer_norm)
            return tfgnn.keras.layers.NextStateFromConcat(mlp)

        num_processors = (1 if shared_processors else num_message_passing_steps)

        processors = []
        for processor_i in range(num_processors):
            processor_name = f"{self.name}/processor_{processor_i}"
            processor_fn_named = functools.partial(processor_fn,
                                                    processor_name=processor_name)
            processors.append(GraphNetworkGraphUpdate(
                edges_next_state_factory=processor_fn_named,
                nodes_next_state_factory=processor_fn_named,
                context_next_state_factory=(processor_fn_named if context_output_size else None),
                reduce_type_to_nodes=reduce_type_to_nodes,
                reduce_type_to_context=reduce_type_to_context,
                use_input_context_state=False, ## ESSAI
                name=processor_name))

        if shared_processors:
            self._processors = processors * num_message_passing_steps
        else:
            self._processors = processors

        # Build graph decoder.
        def decoder_fn(graph_piece, *, edge_set_name=None, node_set_name=None):
            piece_name = (f"edges_{edge_set_name}" if edge_set_name else
                            f"nodes_{node_set_name}" if node_set_name else "context")
            if edge_set_name:
                output_size = edge_output_size
            elif node_set_name:
                output_size = node_output_size
            else:
                output_size = context_output_size
            mlp = build_mlp(num_hidden_layers=num_mlp_hidden_layers,
                            hidden_size=mlp_hidden_size,
                            output_size=output_size,
                            use_layer_norm=False,  # Never LayerNorm for the outputs.
                            name=f"{self.name}/decoder/{piece_name}")
            return mlp(graph_piece[tfgnn.HIDDEN_STATE])

        self._decoder = tfgnn.keras.layers.MapFeatures(
            edge_sets_fn=decoder_fn if edge_output_size else None,
            node_sets_fn=decoder_fn if node_output_size else None,
            context_fn=decoder_fn if context_output_size else None)


    # Call function to update the graphtensor
    def call(self, input_graph: tfgnn.GraphTensor) -> tfgnn.GraphTensor:
        latent_graph = self._encoder(input_graph)
        for processor in self._processors:
            residual_graph = processor(latent_graph)
            latent_graph = sum_graphs(residual_graph, latent_graph)
        output_graph = self._decoder(latent_graph)
        return output_graph

# This is for summing the residual graphs.
# TODO(b/234563300): Consider supporting `MapFeatures` for multiple graphs.
def sum_graphs(graph_1: tfgnn.GraphTensor, graph_2: tfgnn.GraphTensor,
               ) ->  tfgnn.GraphTensor:
    """Sums all features in two identical graphs."""
    assert set(graph_1.edge_sets.keys()) == set(graph_2.edge_sets.keys())
    new_edge_set_features = {}
    for set_name in graph_1.edge_sets.keys():
        new_edge_set_features[set_name] = _sum_feature_dict(
            graph_1.edge_sets[set_name].get_features_dict(),
            graph_2.edge_sets[set_name].get_features_dict())

    assert set(graph_1.node_sets.keys()) == set(graph_2.node_sets.keys())
    new_node_set_features = {}
    for set_name in graph_1.node_sets.keys():
        new_node_set_features[set_name] = _sum_feature_dict(
            graph_1.node_sets[set_name].get_features_dict(),
            graph_2.node_sets[set_name].get_features_dict())

    new_context_features = _sum_feature_dict(
        graph_1.context.get_features_dict(),
        graph_2.context.get_features_dict())
    return graph_1.replace_features(
        edge_sets=new_edge_set_features,
        node_sets=new_node_set_features,
        context=new_context_features)


def _sum_feature_dict(
    features_1: Mapping[str, tf.Tensor],
    features_2: Mapping[str, tf.Tensor]
    ) -> Mapping[str, tf.Tensor]:
    tf.nest.assert_same_structure(features_1, features_2)
    return tf.nest.map_structure(lambda x, y: x + y, features_1, features_2)


def nest_to_numpy(nest):
    return tf.nest.map_structure(lambda x: x.numpy(), nest)

# TODO(b/205123804): Provide a library function for this.
def graph_tensor_spec_from_sample_graph(sample_graph):
    """Build variable node/edge spec given a sample graph without batch axes."""
    tfgnn.check_scalar_graph_tensor(sample_graph)
    sample_graph_spec = sample_graph.spec
    node_sets_spec = {}
    for node_set_name, node_set_spec in sample_graph_spec.node_sets_spec.items():
        new_features_spec = {}
        for feature_name, feature_spec in node_set_spec.features_spec.items():
            new_features_spec[feature_name] = _to_none_leading_dim(feature_spec)
        node_sets_spec[node_set_name] = tfgnn.NodeSetSpec.from_field_specs(
            features_spec=new_features_spec,
            sizes_spec=tf.TensorSpec(shape=(1,), dtype=tf.int32))

    edge_sets_spec = {}
    for edge_set_name, edge_set_spec in sample_graph_spec.edge_sets_spec.items():
        new_features_spec = {}
        for feature_name, feature_spec in edge_set_spec.features_spec.items():
            new_features_spec[feature_name] = _to_none_leading_dim(feature_spec)

        adjacency_spec = tfgnn.AdjacencySpec.from_incident_node_sets(
            source_node_set=edge_set_spec.adjacency_spec.source_name,
            target_node_set=edge_set_spec.adjacency_spec.target_name,
            index_spec=_to_none_leading_dim(edge_set_spec.adjacency_spec.target))

        edge_sets_spec[edge_set_name] = tfgnn.EdgeSetSpec.from_field_specs(
            features_spec=new_features_spec,
            sizes_spec=tf.TensorSpec(shape=(1,), dtype=tf.int32),
            adjacency_spec=adjacency_spec)

    context_spec = sample_graph_spec.context_spec

    return tfgnn.GraphTensorSpec.from_piece_specs(
        node_sets_spec=node_sets_spec,
        edge_sets_spec=edge_sets_spec,
        context_spec=context_spec,
    )

def _to_none_leading_dim(spec):
    new_shape = list(spec.shape)
    new_shape[0] = None
    return tf.TensorSpec(shape=new_shape, dtype=spec.dtype)



### Dataset
[ECMWF ERA5](https://www.ecmwf.int/en/forecasts/dataset/ecmwf-reanalysis-v5)


In [ ]:
def normalizeField(field):
    min = tf.math.reduce_min(field)
    max = tf.math.reduce_max(field)
    return tf.math.divide(
            tf.math.subtract(
                field, 
                min
            ), 
            tf.math.subtract(
                max, 
                min
            )
        ), min, max


def pifoGridGenerator(grib_file: str, 
                      grid:LatLonGrid,
                      **task_kwargs):
    # Load data (8 z500 ggrib_file...)
    ds = xr.load_dataset(grib_file, engine="cfgrib", decode_timedelta=False)
    # Sélection d'une grille réduite avec un sous-échantillonnage (1 point sur 4)
    ds_reduced = ds.isel(latitude=slice(0, None, 4), longitude=slice(0, None, 4))
    
    G_NLat = ds_reduced.latitude.shape[0]
    G_NLon = ds_reduced.longitude.shape[0]
    assert G_NLat==grid.NLat and G_NLon==grid.NLon, "grid definition and grib file mismatch."

    NExamples = ds_reduced.z.shape[0]
    Nfeatures = 3
    NEdgeFeatures = 3
  
    cos_lat, _, _ = normalizeField(tf.math.sin(grid.lat2d*math.pi/180))
    cos_lon, _, _ = normalizeField(tf.math.cos(grid.lon2d*math.pi/180))
    sin_lon, _, _ = normalizeField(tf.math.sin(grid.lon2d*math.pi/180))

    # Create train data
    Z = tf.reshape(tf.convert_to_tensor(ds_reduced.z[0:NExamples]), [NExamples, grid.NNodes])
    Z, _ , _ = normalizeField(Z) # Z/70000 # normalisation à l'arrache
    m = Z.shape[0]

    u = tf.reshape(tf.convert_to_tensor(ds_reduced.u[0:NExamples]), [NExamples, grid.NNodes])
    u, _, _ = normalizeField(u)
    v = tf.reshape(tf.convert_to_tensor(ds_reduced.v[0:NExamples]), [NExamples, grid.NNodes])
    v, _, _ = normalizeField(v)

    #edgeFeatures = tf.zeros([grid.NEdges, NEdgeFeatures], dtype=tf.float32)
    edge_lengthes = tf.convert_to_tensor(grid.lengthes, dtype_hint="float")
    edge_lengthes, _, _ = normalizeField(edge_lengthes)

    x, _, _ = normalizeField(tf.convert_to_tensor(grid.coords_x, dtype_hint="float"))
    y, _, _ = normalizeField(tf.convert_to_tensor(grid.coords_y, dtype_hint="float"))
    edgeFeatures = tf.stack([
            edge_lengthes, 
            x,
            y
        ], axis=1)


    for i in range(m-1): # To allow for making Y
        features = tf.stack([Z[i], u[i], v[i], cos_lat, cos_lon, sin_lon], axis=1) 
        predictions = tf.stack([Z[i+1], u[i+1], v[i+1]], axis=1)
        yield buildGridGNN(features, edgeFeatures, grid), predictions

# TODO : make something cleaner using tfgnn builtin functions ?
def getGraphExample(grid:LatLonGrid):
        # Load data (8 z500 ggrib_file...)
    ds = xr.load_dataset("dataset/202504.grib", engine="cfgrib", decode_timedelta=False)
    #ds_reduced = ds
    # Sélection d'une grille réduite avec un sous-échantillonnage (1 point sur 4)
    ds_reduced = ds.isel(latitude=slice(0, None, 4), longitude=slice(0, None, 4))

    G_NLat = ds_reduced.latitude.shape[0]
    G_NLon = ds_reduced.longitude.shape[0]
    assert G_NLat==grid.NLat and G_NLon==grid.NLon, "grid definition and grib file mismatch."

    NEdgeFeatures = 3

    # Create train data
    NExamples = ds_reduced.z.shape[0]

    Z = tf.reshape(tf.convert_to_tensor(ds_reduced.z[0:NExamples-1]), [NExamples-1, grid.NNodes])
    Z, _ , _ = normalizeField(Z)

    u = tf.reshape(tf.convert_to_tensor(ds_reduced.u[0:NExamples]), [NExamples, grid.NNodes])
    u, _, _ = normalizeField(u)
    v = tf.reshape(tf.convert_to_tensor(ds_reduced.v[0:NExamples]), [NExamples, grid.NNodes])
    v, _, _ = normalizeField(v)

    cos_lat, _, _ = normalizeField(tf.math.sin(grid.lat2d*math.pi/180))
    cos_lon, _, _ = normalizeField(tf.math.cos(grid.lon2d*math.pi/180))
    sin_lon, _, _ = normalizeField(tf.math.sin(grid.lon2d*math.pi/180))

    
    features = tf.stack([Z[0], u[0], v[0], cos_lat, cos_lon, sin_lon], axis=1)

    #Nfeatures = 3
    # edgeFeatures = tf.zeros([grid.NEdges, NEdgeFeatures], dtype=tf.float32)
    edge_lengthes = tf.convert_to_tensor(grid.lengthes, dtype_hint="float")
    edge_lengthes, _, _ =  normalizeField(edge_lengthes)
    x, _, _ = normalizeField(tf.convert_to_tensor(grid.coords_x, dtype_hint="float"))
    y, _, _ = normalizeField(tf.convert_to_tensor(grid.coords_y, dtype_hint="float"))
    
    edgeFeatures = tf.stack([
            edge_lengthes, 
            x,
            y
        ], axis=1)

    # for i in range(10):
    #     ds_reduced.z[i].plot()
    #     plt.savefig("X_"+str(i)+".png", dpi=150, bbox_inches="tight")  # Ajuste la qualité et l'encadrement
    #     plt.close()  
    # ds_reduced.z[1].plot()
    
    return buildGridGNN(features, edgeFeatures, grid), tf.stack([Z[0], u[0], v[0]])

# Quick & dirty way of chaining predictions with ...
def getGraphForFeatures(X, grid:LatLonGrid):
    # Load data (8 z500 ggrib_file...)
    ds = xr.load_dataset("dataset/download.grib", engine="cfgrib", decode_timedelta=False)
    #ds_reduced = ds
    # Sélection d'une grille réduite avec un sous-échantillonnage (1 point sur 4)
    ds_reduced = ds.isel(latitude=slice(0, None, 4), longitude=slice(0, None, 4))

    G_NLat = ds_reduced.latitude.shape[0]
    G_NLon = ds_reduced.longitude.shape[0]
    assert G_NLat==grid.NLat and G_NLon==grid.NLon, "grid definition and grib file mismatch."
    NEdgeFeatures = 3

    #edgeFeatures = tf.zeros([grid.NEdges, NEdgeFeatures], dtype=tf.float32)
    edge_lengthes = tf.convert_to_tensor(grid.lengthes, dtype_hint="float")
    edge_lengthes, _, _  = normalizeField(edge_lengthes)
    x, _, _ = normalizeField(tf.convert_to_tensor(grid.coords_x, dtype_hint="float"))
    y, _, _ = normalizeField(tf.convert_to_tensor(grid.coords_y, dtype_hint="float"))
    edgeFeatures = tf.stack([
            edge_lengthes, 
            x,
            y
        ], axis=1)


    cos_lat, _, _ = normalizeField(tf.math.sin(grid.lat2d*math.pi/180))
    cos_lon, _, _ = normalizeField(tf.math.cos(grid.lon2d*math.pi/180))
    sin_lon, _, _ = normalizeField(tf.math.sin(grid.lon2d*math.pi/180))

    features = tf.concat([X, tf.reshape(cos_lat, (-1, 1)), tf.reshape(cos_lon, (-1, 1)), tf.reshape(sin_lon, (grid.NNodes, -1))], axis=1)
    return buildGridGNN(features, edgeFeatures, grid)


class Era5DatasetProvider(runner.DatasetProvider):
    def __init__(self, grib_file):
        super().__init__()
        self.grib_file = grib_file
        

    def get_dataset(self, context: tf.distribute.InputContext) -> tf.data.Dataset:
        def generator_fn():
            return pifoGridGenerator(self.grib_file)
    
        graph_spec = graph_tensor_spec_from_sample_graph(next(generator_fn()))
        return tf.data.Dataset.from_generator(
            generator_fn, output_signature=graph_spec).repeat()
    
def get_dataset(grib_file, grid:LatLonGrid) -> tf.data.Dataset:
    def generator_fn():
        return pifoGridGenerator(grib_file, grid)

    schema = tfgnn.read_schema("pifo.pbtxt")
    graph_spec = tfgnn.create_graph_spec_from_schema_pb(schema)
    return tf.data.Dataset.from_generator( 
        generator_fn, output_signature=(graph_spec, tf.TensorSpec((grid.NNodes,3), dtype=tf.float32))).repeat()

train_provider = Era5DatasetProvider(grib_file="dataset/download.grib")

### Feature mapping

In [ ]:
denseMapping = tf.keras.layers.Dense(8, "relu")
edge_denseMapping = tf.keras.layers.Dense(4, "relu")
def _set_initial_node_state(node_set, *, node_set_name):
    assert node_set_name == "grid"
    return denseMapping(node_set["features"])
#tf.cast(node_set["features"], tf.float32)
    #
    #return tf.concat(
#        [tf.cast(node_set["features"], tf.float32)[..., None],
#        ],
#        axis=-1)

def _set_initial_edge_state(edge_set, *, edge_set_name):
    assert edge_set_name == "edges"
    return edge_denseMapping(edge_set["edge_features"])
    #return tfgnn.keras.layers.MakeEmptyFeature()(edge_set)

def _set_initial_context_state(context):
    return tfgnn.keras.layers.MakeEmptyFeature()(context)

build_initial_hidden_state = tfgnn.keras.layers.MapFeatures(
    node_sets_fn=_set_initial_node_state,
    edge_sets_fn=_set_initial_edge_state,
    context_fn=_set_initial_context_state)

In [ ]:
#graph, _, _ = getGraphExample()
#gen = pifoGridGenerator("dataset/download.grib")
#graph, _ = next(gen)
#test = build_initial_hidden_state(graph)
# data = tfgnn.keras.layers.Readout(
#                 node_set_name="grid",
#                 feature_name=tfgnn.HIDDEN_STATE
#             )(test)
# data = tfgnn.keras.layers.Readout(
#                 node_set_name="grid",
#                 feature_name="features"
#             )(graph)
# print(data)

#example = tfgnn.write_example(graph)
#schema = tfgnn.create_schema_pb_from_graph_spec(graph.spec)
#tfgnn.write_schema(schema, "pouet.pbtxt")

### Model definition

In [ ]:
def pifo(gtspec: tfgnn.GraphTensorSpec, 
        num_message_passing_steps=1,
        num_mlp_hidden_layers=1,
        mlp_hidden_size=8, 
        latent_size=8,
        output_size=3):
    
    # Input graph from dataset or inference with all input features
    input_graph = tf.keras.layers.Input(type_spec=gtspec)

    # We want to output a difference. So keep track of input features.
    input_state = tfgnn.keras.layers.Readout(
                node_set_name="grid",
                feature_name="features"
            )(input_graph)[:,0:3]

    # Map input features to hidden state
    output_graph = build_initial_hidden_state(input_graph)

    # Apply the encode process decode algorithm
    trainable_gnn = PifoEncodeProcessDecode(
        edge_output_size=None, # No feature output on edge
        node_output_size=output_size,  # Only one feature to predict for now
        context_output_size=None,  # Don't need this output.
        # Other configurable hyperparameters (most combinations should train).
        num_message_passing_steps=num_message_passing_steps,
        num_mlp_hidden_layers=num_mlp_hidden_layers,
        mlp_hidden_size=mlp_hidden_size ,
        latent_size=latent_size,      # taille des features sur chaque noeud
        use_layer_norm=True,
        shared_processors=False,
        )

    output_graph = trainable_gnn(output_graph)

    # Readout final features [num nodes, n features]
    output = tfgnn.keras.layers.Readout(
                node_set_name="grid",
                feature_name=tfgnn.HIDDEN_STATE
            )(output_graph)
    
    # Apply a prediction head to get Y(t) the difference from X(t) to new state
    output = build_mlp(num_hidden_layers=num_mlp_hidden_layers,
        hidden_size=mlp_hidden_size,
        output_size=output_size,
        activate_final=False,
        use_layer_norm=False,  
        name=f"pifo/prediction_head")(output)
    
    # Now we want our output to be X(t) + Y(t)
    output = tf.math.add(input_state, output)
    
    # Now we have a trainable/savable Keras model
    model = tf.keras.Model(input_graph, output)

    return model

### Training

In [ ]:
tf.keras.backend.clear_session()

NUM_ITERATIONS = 10
TRAIN_BATCH_SIZE = 1

def optimizer_fn():
    optimizer = tf.keras.optimizers.Adam(
        learning_rate=tf.keras.optimizers.schedules.ExponentialDecay(
            initial_learning_rate=1e-3,
            decay_steps=1500,
            decay_rate=0.1),
        beta_1=0.9,
        beta_2=0.95
    )
    
    # optimizer = tf.keras.optimizers.Adam(
    #     learning_rate=0.001,
    #     beta_1=0.9,
    #     beta_2=0.95
    # )
    
    return optimizer

schema = tfgnn.read_schema("pifo.pbtxt")
graph_spec = tfgnn.create_graph_spec_from_schema_pb(schema)
grid = LatLonGrid("dataset/202504.grib", 4)
dataset = get_dataset("dataset/202504.grib", grid)
optimizer = optimizer_fn()
loss = tf.keras.losses.MeanSquaredError(reduction="sum_over_batch_size")
metrics = [tf.keras.metrics.RootMeanSquaredError()]
model_checkpoint_callback = tf.keras.callbacks.ModelCheckpoint(
    filepath="./pifo_chk/pifo_{epoch:04d}.weights.h5",
    save_weights_only=True,
    monitor='loss',
    mode='auto',
    save_freq="epoch",
    save_best_only=False)
model = pifo(graph_spec,num_message_passing_steps=4,
        num_mlp_hidden_layers=1,
        mlp_hidden_size=8, # Pour mon petit modèle ça devrait être largement suffisant
        latent_size=8, 
        output_size=3)
model.compile(optimizer = optimizer,
              loss = loss,
              metrics = metrics)
model.summary()
history = model.fit(dataset,
                    batch_size=1,
                    steps_per_epoch=119,
                    epochs=NUM_ITERATIONS,
                    callbacks=[model_checkpoint_callback])

#restored_model = tf.saved_model.load("/home/nicolas/Projects/pifocast/model/export")
#model = run_result.trained_model
#print(model)

### Model training metrics

In [ ]:
# @title Plot learning curves. { form-width: "30%" }

smoooth_window_half_width = 3

# For plotting learning curves.
losses_curve = history.history['loss']
scores_curve = history.history['root_mean_squared_error'] 

plt.plot(losses_curve)
plt.title('model loss')
plt.ylabel('loss')
plt.xlabel('epoch')
plt.show()

plt.plot(losses_curve)
plt.title('model score')
plt.ylabel('score')
plt.xlabel('epoch')
plt.show()

### Model saving / loading

In [ ]:
#model.save_weights('pifo_chk/final_checkpoint')


In [ ]:
# schema = tfgnn.read_schema("pifo.pbtxt")
# graph_spec = tfgnn.create_graph_spec_from_schema_pb(schema)
# grid = LatLonGrid("dataset/202504.grib", 4)
# model = pifo(graph_spec,num_message_passing_steps=4,
#         num_mlp_hidden_layers=1,
#         mlp_hidden_size=8, # Pour mon petit modèle ça devrait être largement suffisant
#         latent_size=8, 
#         output_size=3)
# model.load_weights('./pifo_chk/final_checkpoint')


### Model saving / loading

In [ ]:
def plot(Y, width, height, fileName):
    img = tf.reshape(Y, [height, width])
    plt.imshow(img, interpolation='none')
    plt.show()
    plt.imsave(fileName, img)


graph, X = getGraphExample(grid)
Y = model(graph)

plot(X[0], grid.NLon, grid.NLat, "images/Y_0.png")


#print(G_NLon, G_NLat)
plot(Y[:,0], grid.NLon, grid.NLat, "images/Y_1.png")


for i in range(2,20):    
    graph = getGraphForFeatures(Y, grid)
    Y = model(graph)
    plot(Y[:,0], grid.NLon, grid.NLat, "images/Y_"+str(i)+".png")
